In [1]:
import pandas as pd
import plotly.graph_objects as go

In [2]:
def load_impacto_mensal(proposta:str)->pd.DataFrame:
    fname = f'impacto_mensal_{proposta}.csv'
    return pd.read_csv(fname, index_col=0, sep=';')

In [3]:
impactos_mensais ={
    'Equiparação com a tabela dos AMCI' : load_impacto_mensal('amci'),
    'Reajuste pelo IPC-Fipe - jun. 2016' : load_impacto_mensal('ipc'),
    'Reajuste pelo IPC-Fipe - jun. 2021' : load_impacto_mensal('ipc_nunes'),
    "Situação atual" : load_impacto_mensal('amci')[['nivel_carreira', 'valor_total_prefeitura_atual']].rename({'valor_total_prefeitura_atual': 'valor_total_prefeitura_proposta'}, axis=1)
}

In [4]:
impactos_mensais['Equiparação com a tabela dos AMCI']

,nivel_carreira,valor_total_prefeitura_atual,id_proposta_atual,valor_total_prefeitura_proposta,id_proposta_proposta,impacto_mensal
0,1,1236354.23,situacao_atual,1481375.81,unificacao_amci,245021.58
1,2,975355.80,situacao_atual,1127910.00,unificacao_amci,152554.20
2,3,35419.96,situacao_atual,41203.64,unificacao_amci,5783.68
3,4,19257.07,situacao_atual,22493.76,unificacao_amci,3236.69
4,5,312637.19,situacao_atual,363465.00,unificacao_amci,50827.81
5,6,732991.09,situacao_atual,863539.30,unificacao_amci,130548.21


In [5]:

def gerar_grafico_niveis_df(df, col_valor:str, titulo:str, nome_arquivo="grafico_niveis.png"):
    
    col_nivel = 'nivel_carreira'
    
    cores = ["steelblue"] * len(df)
    indice_maximo = df[col_valor].idxmax()
    cores[indice_maximo] = "crimson"

    # Formatação para o texto sobre as barras (R$ 1.234,56)
    texto_formatado = df[col_valor].apply(
        lambda x: f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    )

    fig = go.Figure(data=[
        go.Bar(
            x=df[col_nivel],
            y=df[col_valor],
            text=texto_formatado,
            textposition="outside",
            marker_color=cores
        )
    ])

    fig.update_layout(
        title=dict(
            text=titulo,
            x=0.5,
            xanchor="center",
            font=dict(size=22)
        ),
        # Define os separadores globalmente: o primeiro é o decimal, o segundo é o de milhar
        separators=",.",
        plot_bgcolor="white",
        width=1200,
        height=600,
        xaxis=dict(
            title="Níveis de Carreira",
            tickangle=-45,
            categoryorder="array",
            categoryarray=df[col_nivel].tolist()
        ),
        yaxis=dict(
            title="Valores em R$",
            showgrid=True,
            gridcolor="lightgrey",
            # Formata os números do eixo Y com separador de milhar e 2 casas decimais
            tickformat=",2f"
        ),
        margin=dict(l=50, r=50, t=100, b=120)
    )

    fig.write_image(nome_arquivo)

    return fig

In [6]:
for nome, df in impactos_mensais.items():
    titulo = f"Custo total por nível: {nome}"
    nome_arquivo = f"grafico_niveis_{nome.replace(' ', '_').lower()}.png"
    gerar_grafico_niveis_df(df, "valor_total_prefeitura_proposta", titulo, nome_arquivo)

In [7]:
for proposta, df in impactos_mensais.items():
    valor_total = df['valor_total_prefeitura_proposta'].sum()*12
    valor_formatado = f"R$ {valor_total:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    print(f"Valor total anual: {valor_formatado} - {proposta}")

Valor total anual: R$ 46.799.850,12 - Equiparação com a tabela dos AMCI
Valor total anual: R$ 43.253.796,72 - Reajuste pelo IPC-Fipe - jun. 2016
Valor total anual: R$ 39.744.184,08 - Reajuste pelo IPC-Fipe - jun. 2021
Valor total anual: R$ 39.744.184,08 - Situação atual


In [8]:
for proposta, df in impactos_mensais.items():
    try:
        valor_total = df['impacto_mensal'].sum()*12
        valor_formatado = f"R$ {valor_total:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
        print(f"Impacot total anual: {valor_formatado} - {proposta}")
    except KeyError:
        print(proposta, "não possui impacto")

Impacot total anual: R$ 7.055.666,04 - Equiparação com a tabela dos AMCI
Impacot total anual: R$ 3.509.612,64 - Reajuste pelo IPC-Fipe - jun. 2016
Impacot total anual: R$ 0,00 - Reajuste pelo IPC-Fipe - jun. 2021
Situação atual não possui impacto


In [9]:
niveis = pd.read_csv('tabelas_vencimentos_utilizadas.csv', sep=';', index_col=0)

In [10]:
niveis

,nome_tabela,nivel,vencimento
0,atual,1,13544.95
1,atual,2,14899.45
2,atual,3,15271.93
3,atual,4,15653.72
4,atual,5,16045.07
...,...,...,...
10,original_atualizada_ipc_nunes,11,21984.11
11,original_atualizada_ipc_nunes,12,24182.51
12,original_atualizada_ipc_nunes,13,25270.72
13,original_atualizada_ipc_nunes,14,26407.91


In [11]:
import plotly.graph_objects as go
import pandas as pd

def exportar_graficos_comparativos(df, col_valor="vencimento"):
    col_nivel = "nivel"
    col_tabela = "nome_tabela"
    
    # Identifica as tabelas que serão comparadas com a 'atual'
    tabelas_extras = [t for t in df[col_tabela].unique() if t != "atual"]
    
    for tabela in tabelas_extras:
        # Filtra apenas o par necessário
        df_par = df[df[col_tabela].isin(["atual", tabela])]
        
        fig = go.Figure()

        # Adiciona as barras para 'atual' e para a 'tabela' da vez
        for nome in ["atual", tabela]:
            df_sub = df_par[df_par[col_tabela] == nome]
            
            # Formatação Real (sem centavos para evitar sobreposição de texto)
            texto = df_sub[col_valor].apply(
                lambda x: f"R$ {x:,.0f}".replace(",", "X").replace(".", ",").replace("X", ".")
            )

            fig.add_trace(go.Bar(
                x=df_sub[col_nivel],
                y=df_sub[col_valor],
                name=nome.replace("_", " ").title(),
                text=texto,
                textposition="outside",
                marker_color="steelblue" if nome == "atual" else "crimson"
            ))

        fig.update_layout(
            title=dict(
                text=f"Comparativo: Atual vs {tabela.replace('_', ' ').title()}",
                x=0.5,
                font=dict(size=22)
            ),
            barmode="group",
            separators=",.",
            plot_bgcolor="white",
            width=1200,
            height=600,
            xaxis=dict(title="Nível de Carreira", type="category"),
            yaxis=dict(
                title="Vencimento (R$)",
                showgrid=True,
                gridcolor="lightgrey",
                tickformat=",0f"
            ),
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
            margin=dict(l=50, r=50, t=100, b=50)
        )

        # Salva cada gráfico com o nome da tabela correspondente
        nome_arquivo = f"comparativo_atual_vs_{tabela}.png"
        fig.write_image(nome_arquivo, engine="kaleido")
        print(f"Arquivo salvo: {nome_arquivo}")

# Exemplo de uso:
# exportar_graficos_comparativos(df)

In [12]:
exportar_graficos_comparativos(niveis)

/tmp/ipykernel_64657/3595305641.py:59: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  fig.write_image(nome_arquivo, engine="kaleido")


Arquivo salvo: comparativo_atual_vs_original.png


/tmp/ipykernel_64657/3595305641.py:59: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  fig.write_image(nome_arquivo, engine="kaleido")


Arquivo salvo: comparativo_atual_vs_amci.png


/tmp/ipykernel_64657/3595305641.py:59: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  fig.write_image(nome_arquivo, engine="kaleido")


Arquivo salvo: comparativo_atual_vs_original_atualizada_ipc.png


/tmp/ipykernel_64657/3595305641.py:59: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  fig.write_image(nome_arquivo, engine="kaleido")


Arquivo salvo: comparativo_atual_vs_original_atualizada_ipc_nunes.png


In [13]:
impacto = pd.read_csv('impactos_anuais.csv', sep=';', index_col=0)

In [14]:
impacto

,nivel,vencimento
0,reajuste_ipc_gestao_nunes,0.00
1,reajuste_ipc,3509612.64
2,reajuste_amci,7055666.04


In [15]:
impacto= impacto.rename({'nivel' : 'Proposta Reajusta', 'vencimento' : "Impacto orçamentário anualizado"}, axis=1)

In [16]:
impacto

,Proposta Reajusta,Impacto orçamentário anualizado
0,reajuste_ipc_gestao_nunes,0.00
1,reajuste_ipc,3509612.64
2,reajuste_amci,7055666.04


In [17]:
impacto['Proposta Reajusta'] = impacto['Proposta Reajusta'].str.replace('_', ' ').str.replace('ipca', 'icp fipe').str.title()

In [18]:
impacto

,Proposta Reajusta,Impacto orçamentário anualizado
0,Reajuste Ipc Gestao Nunes,0.00
1,Reajuste Ipc,3509612.64
2,Reajuste Amci,7055666.04


In [19]:
import plotly.graph_objects as go
import pandas as pd

def gerar_grafico_simples_ordenado(df, col_valor="vencimento", titulo="Comparativo de Reajustes", nome_arquivo="grafico_simples.png"):
    
    col_tabela = "Proposta Reajusta"
    
    # Ordena o DataFrame pelo valor do vencimento (menor para o maior)
    df_ordenado = df.sort_values(by=col_valor, ascending=True)

    # Formatação para o texto sobre as barras (R$ 1.234)
    texto_formatado = df_ordenado[col_valor].apply(
        lambda x: f"R$ {x:,.0f}".replace(",", "X").replace(".", ",").replace("X", ".")
    )

    # Define cores: Azul para a 'atual' e Steelblue para as demais
    cores = ["steelblue" if n != "atual" else "darkblue" for n in df_ordenado[col_tabela]]

    fig = go.Figure(data=[
        go.Bar(
            x=df_ordenado[col_tabela],
            y=df_ordenado[col_valor],
            text=texto_formatado,
            textposition="outside",
            marker_color=cores
        )
    ])

    fig.update_layout(
        title=dict(
            text=titulo,
            x=0.5,
            font=dict(size=22)
        ),
        separators=",.",
        plot_bgcolor="white",
        width=1000,
        height=600,
        xaxis=dict(
            title="Cenários / Tabelas",
            tickangle=0 # Mantém reto para facilitar leitura se forem poucos nomes
        ),
        yaxis=dict(
            title="Vencimento (R$)",
            showgrid=True,
            gridcolor="lightgrey",
            tickformat=",0f"
        ),
        margin=dict(l=50, r=50, t=100, b=100)
    )

    fig.write_image(nome_arquivo, engine="kaleido")

    return fig

In [20]:
gerar_grafico_simples_ordenado(impacto, col_valor='Impacto orçamentário anualizado')

/tmp/ipykernel_64657/2206286243.py:52: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  fig.write_image(nome_arquivo, engine="kaleido")


In [22]:
situacao_atual = pd.read_csv('situacao_atual.csv', sep=';', index_col=0)

In [23]:
situacao_atual.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,vale_refeicao,contribuicao_iprem,contribuicao_inss,previdencia_complementar,irpf_na_fonte,valor_total_prefeitura
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG2,SMC,2021-10-01,2,False,14899.45,1241.62,413.87,325.25,664.84,0.0,1779.87,574.91,3643.88,16255.93
1,7575491,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,False,13544.95,1128.75,376.25,325.25,664.84,0.0,1779.87,464.86,3230.01,15054.76
2,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,False,14899.45,1241.62,413.87,325.25,664.84,0.0,1779.87,574.91,3643.88,16255.93
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,False,14899.45,1241.62,413.87,325.25,664.84,0.0,1779.87,574.91,3643.88,16255.93
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,True,16045.07,1337.09,445.70,325.25,664.84,4867.0,0.00,0.00,3993.93,19691.02


In [26]:
situacao_atual['nome'].str.startswith('recem_nomeado').sum()

np.int64(13)

In [31]:
qtd_por_nivel = situacao_atual.groupby('nivel_carreira').count()[['rf']].rename({'rf' : 'Quantidade'}, axis=1)

In [34]:
import plotly.graph_objects as go

fig = go.Figure(data=[
    go.Bar(
        x=qtd_por_nivel.index.astype(str),  # Convertendo o índice para string para melhor exibição
        y=qtd_por_nivel['Quantidade'],
        text=qtd_por_nivel['Quantidade'],
        textposition='outside',
        marker_color='steelblue'
    )
])

fig.update_layout(
    title=dict(
        text="Quantidade por Nível na Carreira",
        x=0.5,
        font=dict(size=22)
    ),
    xaxis=dict(
        title="Nível na Carreira",
        tickangle=0
    ),
    yaxis=dict(
        title="Quantidade",
        showgrid=True,
        gridcolor="lightgrey"
    ),
    plot_bgcolor="white",
    width=800,
    height=500,
    margin=dict(l=50, r=50, t=100, b=50)
)
fig.write_image('quantidade_pessoas_por_nivel.png', engine="kaleido")

fig.show()

/tmp/ipykernel_64657/2642592332.py:33: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  fig.write_image('quantidade_pessoas_por_nivel.png', engine="kaleido")
